In [2]:
! pip install ultralytics opencv-python pyttsx3


In [3]:
import cv2
import time
import pyttsx3
from ultralytics import YOLO

# =========================
# INITIALIZE VOICE ENGINE
# =========================
engine = pyttsx3.init()
engine.setProperty('rate', 165)

def speak(text):
    engine.say(text)
    engine.runAndWait()

# =========================
# LOAD YOLO MODEL
# =========================
model = YOLO("best.pt")  # path to your trained model

# =========================
# OPEN FRONT CAMERA
# =========================
cap = cv2.VideoCapture(0)

if not cap.isOpened():
    print("❌ Camera not accessible")
    exit()

print("✅ Camera started")

# =========================
# ALERT CONTROL
# =========================
last_alert_time = 0
ALERT_DELAY = 3  # seconds

# =========================
# MAIN LOOP
# =========================
while True:
    ret, frame = cap.read()
    if not ret:
        break

    # Run YOLO inference
    results = model(frame, conf=0.4, verbose=False)

    helmet_detected = False
    vest_detected = False

    for r in results:
        for box in r.boxes:
            cls_id = int(box.cls[0])
            class_name = model.names[cls_id]

            x1, y1, x2, y2 = map(int, box.xyxy[0])

            if class_name.lower() == "helmet":
                helmet_detected = True
                color = (0, 255, 0)
            elif class_name.lower() == "vest":
                vest_detected = True
                color = (255, 255, 0)
            else:
                color = (0, 0, 255)

            cv2.rectangle(frame, (x1, y1), (x2, y2), color, 2)
            cv2.putText(frame, class_name, (x1, y1 - 10),
                        cv2.FONT_HERSHEY_SIMPLEX, 0.6, color, 2)

    current_time = time.time()

    # =========================
    # DECISION LOGIC
    # =========================
    if helmet_detected and vest_detected:
        status_text = "APPROVED - PPE OK"
        status_color = (0, 255, 0)

    else:
        status_text = "ALERT - PPE MISSING"
        status_color = (0, 0, 255)

        if current_time - last_alert_time > ALERT_DELAY:
            if not helmet_detected and not vest_detected:
                speak("Alert. Helmet and safety vest missing.")
            elif not helmet_detected:
                speak("Alert. Helmet not detected.")
            elif not vest_detected:
                speak("Alert. Safety vest not detected.")

            last_alert_time = current_time

    # =========================
    # DISPLAY STATUS
    # =========================
    cv2.rectangle(frame, (0, 0), (frame.shape[1], 50), (0, 0, 0), -1)
    cv2.putText(frame, status_text, (20, 35),
                cv2.FONT_HERSHEY_SIMPLEX, 1, status_color, 3)

    cv2.imshow("PPE Detection - Live Camera", frame)

    # Press Q to quit
    if cv2.waitKey(1) & 0xFF == ord('q'):
        break

# =========================
# CLEANUP
# =========================
cap.release()
cv2.destroyAllWindows()
engine.stop()


✅ Camera started


KeyboardInterrupt: 